### Understanding compressibility for 1D XYZ Hamiltonian

$$
{\hat{\mathcal{H}}_{XYZ} = \sum_{j} (J_{x} \sigma_j^{x} \sigma_{j+1}^{x} + J_{y} \sigma_j^{y} \sigma_{j+1}^{y} + J_{z} \sigma_j^{z} \sigma_{j+1}^{z}) + \sum_{j} (h_{x} \sigma_j^{x} + h_{y} \sigma_j^{y} + h_{z} \sigma_j^{z})}
$$

In [ ]:
from qiskit import QuantumCircuit
from qiskit.transpiler import CouplingMap
from qiskit.synthesis import SuzukiTrotter, LieTrotter

from qiskit_addon_utils.problem_generators import (
    generate_time_evolution_circuit,
    generate_xyz_hamiltonian
)

import numpy as np
import scipy

In [ ]:
num_qubits = 80 # increase the number to the largest linear chain you can get in the IBM Quantum Computer
Jx, Jy, Jz = np.pi/8, np.pi/4, np.pi/2 # feel free to change these parameters and see how the results change
h_x, h_y, h_z = np.pi/3, np.pi/6, np.pi/9 # feel free to change these parameters and see how the results change
dt = 0.01 # time evolution of each trotter step; feel free to increase or decrease and see how the results change
num_trotter_steps = 10 # decide on the number of trotter steps; do some trial and error to find the 2-qubit depth of the circuit for different trotter steps

Build a linear coupling map and construct the Hamiltonian from it

In [ ]:
coupling_map = CouplingMap.from_line(num_qubits)

In [ ]:
hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(Jx, Jy, Jz),
    ext_magnetic_field=(h_x, h_y, h_z),
)
print(hamiltonian)

Construct a half-filled initial state (also called Neel State). Note a Neel state looks like 010101...

In [ ]:
init_state_neel = QuantumCircuit(num_qubits)
for i in range(num_qubits):
    if i%2 == 0:
        init_state_neel.x(i)

Create Hamiltonian simulation circuit with Lie Trotter decomposition

In [ ]:
circuit = generate_time_evolution_circuit(
    hamiltonian,
    time=dt*num_trotter_steps, # total time of evolution
    synthesis=LieTrotter(reps=num_trotter_steps), 
)

Compose this circuit with the initial state

In [ ]:
circuit = init_state_neel.compose(circuit)

Let us select our observable as $O = \frac{1}{n}\sum_i Z_i$ acting on each qubit

In [ ]:
from qiskit.quantum_info import SparsePauliOp
observable = SparsePauliOp(['I'*i + 'Z' + 'I'*(num_qubits-i-1) for i in range(num_qubits)], 
                            coeffs=[1/num_qubits]*num_qubits)
observable

#### Compress the circuit using AQC

Here we are providing some guides to implement AQC. You should look into [this tutorial](https://quantum.cloud.ibm.com/docs/en/tutorials/approximate-quantum-compilation-for-time-evolution) for more details on AQC

In [ ]:
from qiskit_addon_aqc_tensor.ansatz_generation import (
    generate_ansatz_from_circuit,
)
from qiskit_addon_aqc_tensor.objective import MaximizeStateFidelity
from qiskit_addon_aqc_tensor.simulation.quimb import QuimbSimulator
from qiskit_addon_aqc_tensor.simulation import tensornetwork_from_circuit
from qiskit_addon_aqc_tensor.simulation import compute_overlap

In [ ]:
aqc_trotter_steps = 1 # trying to compress 10 trotter steps into 1

We need to ensure that the time of evolution does not change even when the number of trotter steps is lowered for AQC. The goal is to try to compress the circuit to do the same time of evolution in fewer steps.

In [ ]:
aqc_circuit = generate_time_evolution_circuit(
    hamiltonian,
    time=dt*num_trotter_steps, # total time of evolution
    synthesis=LieTrotter(reps=aqc_trotter_steps), 
)

**Q1**: Execute the original circuit and the aqc circuit and plot the expectation values

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2
service = QiskitRuntimeService()
backend = service.least_busy(min_num_qubits=127)
backend

In [ ]:
from qiskit import generate_preset_pass_manager
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

isa_circuit = # transpile the circuit
isa_aqc_circuit = # transpile the aqc circuit
isa_observable = # apply the layout to the observable

In [ ]:
estimator = EstimatorV2(mode=backend)
pubs = [(isa_circuit, isa_observable), (isa_aqc_circuit, isa_observable)]
job = estimator.run(pubs)

In [ ]:
result = job.result()[0]
aqc_result = job.result()[1]

Obtain the expectation value and show them as a bar plot

##### Now to improve the compressed circuit

In [ ]:
### Create an ansatz from the circuit

aqc_ansatz, aqc_initial_parameters = generate_ansatz_from_circuit(
    aqc_circuit
)

Note the depth of the ansatz circuit, and the number of parameters. We need to find optimal values for those parameters so that the overlap of the compressed circuit with the actual circuit is maximized

In [ ]:
print(f"Target circuit:         depth {circuit.depth()}")
print(
    f"Ansatz circuit:         depth {aqc_ansatz.depth()}, with {len(aqc_initial_parameters)} parameters"
)

In [ ]:
import quimb.tensor
simulator_settings = QuimbSimulator(
    quimb.tensor.CircuitMPS, autodiff_backend="jax"
)

In [ ]:
### We create a Matrix Product State (MPS) version of the actual circuit, which we call the target. The overlap has to be maximized with this circuit

aqc_target_mps = tensornetwork_from_circuit(
    circuit, simulator_settings
)
print("Target MPS maximum bond dimension:", aqc_target_mps.psi.max_bond())

In [ ]:
init_mps = tensornetwork_from_circuit(aqc_circuit, simulator_settings)

# we calculate the overlap of the current AQC circuit with the actual circuit of interest.
# note how little the overlap is: which explains the difference in the hardware result you observed before
starting_fidelity = abs(compute_overlap(init_mps, aqc_target_mps)) ** 2
print("Starting fidelity:", starting_fidelity)

Now we shall optimize the parameters. We shall try to reach an overlap of 99%. But note that the required overlap can vary from problem to problem

In [ ]:
from scipy.optimize import minimize
import datetime

# Setting values for the optimization
aqc_stopping_fidelity = 0.99
aqc_max_iterations = 200 # how many times the variational method will run to optimize the parameters

stopping_point = 1.0 - aqc_stopping_fidelity
objective = MaximizeStateFidelity(
    aqc_target_mps, aqc_ansatz, simulator_settings
)

def callback(intermediate_result):
    fidelity = 1 - intermediate_result.fun
    print(
        f"{datetime.datetime.now()} Intermediate result: Fidelity {fidelity:.8f}"
    )
    if intermediate_result.fun < stopping_point:
        # Good enough for now
        raise StopIteration


result = minimize(
    objective,
    aqc_initial_parameters,
    method="L-BFGS-B",
    jac=True,
    options={"maxiter": aqc_max_iterations},
    callback=callback,
)
if (
    result.status
    not in (
        0,
        1,
        99,
    )
):  # 0 => success; 1 => max iterations reached; 99 => early termination via StopIteration
    raise RuntimeError(
        f"Optimization failed: {result.message} (status={result.status})"
    )

print(f"Done after {result.nit} iterations.")
aqc_final_parameters = result.x

Note the status by printing the result. It was unable to find parameters that allow 99% overlap. This is because we are going for too much compression. Compressing 10 trotter steps to 1 is not feasible. So we need to increase the number of trotter steps we are compressing into

In [ ]:
result

**Q2**: Increase the number of trotter steps to 2, 3... and find the minimum number of trotter steps required such that after optimizing the parameters the overlap between the AQC circuit and the actual circuit is $\geq$ 99%. For that case, run the original circuit and the compressed optimized circuit on the hardware, and show the expectation values.

**Q3**: At 80 qubits, the ideal expectation values are not known. One method to simulate quantum circuits is Pauli Propagation: https://github.com/Qiskit/pauli-prop. In this part, you will implement the same problem in Pauli Propagation; and compare the result as well as execution time with QPU. In particular, make use of [this](https://github.com/Qiskit/pauli-prop/blob/main/docs/tutorials/01_classically_estimate_expectation_values.ipynb) tutorial. Plot the ideal expectation value obtained from Pauli Propagation, and the computed expectation values from the original and the compressed circuits.

**Q4**: Next, we shall understand which direction of the external magnetic field makes the circuit more difficult to compress. To understand this, we shall methodically remove one or more values of the $h$, and repeat the experiment. Let us start with removing $h_z$. This implies that the external magnetic field is aligned along $h_x + h_y$ plane.

***Why is this excercise useful?*** Note that if you can compress 10 trotter steps to, say, $4$ for a particular case, and $3$ for another case, it implies that the circuit in the second case is easier to compress -- which means this circuit can be easily simulated by a tensor network, and has lesser chances of being a candidate for quantum advantage. This simple set of experiments allow you to decipher such a strong understanding.

In [ ]:
hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(Jx, Jy, Jz),
    ext_magnetic_field=(h_x, h_y, 0), # h_z is set to 0
)
print(hamiltonian)

Repeat the above experiment and show how many trotter steps are required to compress this circuit using AQC (Note that you should also set $h_z = 0$ when creating the AQC circuit).

Now repeat the experiment by removing $h_x$ and $h_y$ values. This implies that the external magnetic field is along $h_z$ direction only.

**Q5**: *Using error mitigation*: The results from the hardware are noisy, and therefore may not be perfectly reliable. But we can use error mitigation to account for the noise and take the results from the original and the compressed circuit closer to the ideal. We shall use Probabilistic Error Amplification (PEA) and Twirled Readout Error Extinction (TREX) for this experiment. The first one accounts for Gate Errors, while the second one accounts for SPAM error.

For PEA, it is necessary to first learn the noise in the system. This can be done via NoiseLearner: https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/noise-learner-noise-learner. Also look into this tutorial: https://quantum.cloud.ibm.com/docs/en/tutorials/probabilistic-error-amplification#learn-the-noise-model-for-pea

In [ ]:
### Learn the noise in the system by running NoiseLearner

In [ ]:
### Run the circuit with PEA and TREX error mitigations and calculate the probability of occupancy for each qubit

**Q6**: *Using good noise factors for PEA*

A general issue with PEA is that if you use any random noise factor, the result may not improve. Therefore, it is important to understand which noise factors and extrapolators are good for the given circuit. A method to do that is to cliffordize the circuit, calculate the ideal result for the clifford circuit, try out different noise factors on the clifford circuit and select the one with the best result.

This can be performed using the NEAT tool: https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/debug-tools-neat

In [ ]:
#### Cliffordize the Hamiltonian simulation circuit

In [ ]:
### Calculate the ideal expectation value for the clifford circuit

In [ ]:
### Test for different noise factors and extrapolators
### Test with (1,3,5), (1,2,3), (1,1.2,1.4), (1,1.1,1.2) and for extrapolators linear, quadratic and exponential
### Find the noise factor and extrapolator that best matches the ideal expectation value for the clifford circuit

**NOTE** Since the depth of the original and the AQC compressed circuits are different, the optimal noise factors may not be the same for the two. You need to run the above experiment separately for the two cases to find the best noise factors for each one of them.

In [ ]:
### Use the best noise factor and extrapolator to calculate the mitigated expectation values for the original and compressed cases
### Plot the ideal, noisy and mitigated expectation values for the original and compressed cases